In [ ]:
# =========================================
# H4. 가설 데이터 전처리
# 목적:
# 1) 설문 원본(survey_response.csv) 로드
# 2) H4(리저브 미방문 이유) 분석용 컬럼 정리/표준화
# 3) 미방문 집단(74명) 전용 데이터셋 생성
# 4) 카이제곱/비율검정에 쓰기 좋게 이유 카테고리 통합
# 5) CSV 저장
# =========================================

import os
import pandas as pd
import numpy as np

# -----------------------------------------
# 0. 데이터 로드
# -----------------------------------------

# 현재 업로드 파일 기준 (여기 환경)
file_path = r"C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data\survey_response.csv"

survey = pd.read_csv(file_path)
print("원본 shape:", survey.shape)
print("컬럼 목록:", survey.columns.tolist())

# -----------------------------------------
# 1. 핵심 컬럼만 확인 (H4 관련)
# -----------------------------------------
h4_cols = [
    "reserve_awareness_flag",      # 리저브 매장 인지 여부 (Yes/No)
    "reserve_awareness_channel",   # 리저브를 알게 된 경로
    "reserve_visited_flag",        # 리저브 방문 여부 (Yes/No)
    "reserve_nonvisit_reason",     # 미방문 이유
    "reserve_improvement"          # 개선 필요 항목
]

missing_cols = [c for c in h4_cols if c not in survey.columns]
if missing_cols:
    raise KeyError(f"H4 분석에 필요한 컬럼 누락: {missing_cols}")

# -----------------------------------------
# 2. 값 표준화 (Yes/No -> 한글, 공백 제거)
# -----------------------------------------
df = survey.copy()

for c in h4_cols:
    df[c] = df[c].astype(str).str.strip()

# 문자열 "nan" 정리
df = df.replace({"nan": np.nan, "": np.nan})

# Yes/No 한글 통일
yn_map = {
    "Yes": "예",
    "No": "아니오",
    "yes": "예",
    "no": "아니오"
}
df["reserve_awareness_flag"] = df["reserve_awareness_flag"].replace(yn_map)
df["reserve_visited_flag"] = df["reserve_visited_flag"].replace(yn_map)

# -----------------------------------------
# 3. H4 분석용 파생변수 생성
# -----------------------------------------

# (1) 미방문 여부 플래그
df["is_nonvisit"] = (df["reserve_visited_flag"] == "아니오").astype(int)

# (2) 인지 집단 라벨 (방문 여부와 결합해서 보기에 좋게)
# - H4에서 특히 중요한 그룹: "몰라서 미방문", "알고도 미방문"
df["h4_group"] = np.select(
    [
        (df["reserve_visited_flag"] == "아니오") & (df["reserve_awareness_flag"] == "아니오"),
        (df["reserve_visited_flag"] == "아니오") & (df["reserve_awareness_flag"] == "예"),
        (df["reserve_visited_flag"] == "예")
    ],
    [
        "몰라서 미방문",
        "알고도 미방문",
        "방문 경험 있음"
    ],
    default="기타"
)

# (3) 미방문 이유 카테고리 통합 (카이제곱 검정용)
# 원본 이유가 너무 세분화되면 일부 셀 빈도가 작아서 검정 안정성이 떨어질 수 있음.
# 따라서 H4 해석 목적에 맞게 상위 카테고리로 묶어줌.
reason_map = {
    "리저브 매장 존재여부 미인지(몰랐음)": "인지/정보 부족",
    "접근성 불편": "접근성",
    "가격 부담": "가격",
    "커피 취향/관심 낮음(단순 카페인 수혈 - 일반 매장으로 충분)": "관심/니즈 낮음",
    "매장 분위기/인테리어 불만족": "기타 경험요인"
}

df["nonvisit_reason_grouped"] = df["reserve_nonvisit_reason"].replace(reason_map)

# (4) 개선 필요 항목 통합 (H4-2용)
# reserve_improvement가 카테고리 개수가 너무 많거나 빈도 낮은 값이 있으면 통합
improve_map = {
    "매장 접근성 개선(추가 매장 확대)": "접근성 개선",
    "가격 인하": "가격 개선",
    "혼잡 및 대기 개선(예약/대기 시스템)": "혼잡/대기 개선",
    "리저브 바(Bar) 1:1 서비스 부담 완화": "서비스 부담 완화"
}
df["improve_grouped"] = df["reserve_improvement"].replace(improve_map)

# -----------------------------------------
# 4. H4 핵심 서브셋 생성
# -----------------------------------------

# (A) 전체 H4용 (원본 + 파생변수)
h4_all = df.copy()

# (B) 미방문자 74명 전용
h4_nonvisit = df[df["reserve_visited_flag"] == "아니오"].copy()

# (C) 미방문자 중, H4-2 비교용
# - "몰라서 미방문" vs "알고도 미방문" 두 집단만 남김
h4_nonvisit_compare = h4_nonvisit[h4_nonvisit["h4_group"].isin(["몰라서 미방문", "알고도 미방문"])].copy()

# -----------------------------------------
# 5. QC(검문소) 출력
# -----------------------------------------
print("\n[QC] 방문 여부 분포")
print(h4_all["reserve_visited_flag"].value_counts(dropna=False))

print("\n[QC] H4 그룹 분포")
print(h4_all["h4_group"].value_counts(dropna=False))

print("\n[QC] 미방문자 수 (기대: 74)")
print(len(h4_nonvisit))

print("\n[QC] 미방문 이유 분포 (원본)")
print(h4_nonvisit["reserve_nonvisit_reason"].value_counts(dropna=False))

print("\n[QC] 미방문 이유 분포 (통합)")
print(h4_nonvisit["nonvisit_reason_grouped"].value_counts(dropna=False))

print("\n[QC] 개선 필요 항목 분포 (미방문자 기준)")
print(h4_nonvisit["improve_grouped"].value_counts(dropna=False))

# -----------------------------------------
# 6. 저장
# -----------------------------------------
# 로컬 저장 경로 (본인 환경 기준)
save_dir = r"C:\Users\user\Desktop\데이터 분석 7기 자료\파이널프로젝트\멋사 Final_Project_env\data"

# 여기 환경에서 테스트할 때는 /mnt/data 사용 가능
# save_dir = "/mnt/data"

os.makedirs(save_dir, exist_ok=True)

h4_all.to_csv(os.path.join(save_dir, "survey_h4_refined.csv"), index=False, encoding="utf-8-sig")
h4_nonvisit.to_csv(os.path.join(save_dir, "survey_h4_nonvisit.csv"), index=False, encoding="utf-8-sig")
h4_nonvisit_compare.to_csv(os.path.join(save_dir, "survey_h4_nonvisit_compare.csv"), index=False, encoding="utf-8-sig")

print("\n✅ 저장 완료")
print(os.path.join(save_dir, "survey_h4_refined.csv"))
print(os.path.join(save_dir, "survey_h4_nonvisit.csv"))
print(os.path.join(save_dir, "survey_h4_nonvisit_compare.csv"))

원본 shape: (200, 16)
컬럼 목록: ['gender', 'age_group', 'region', 'cafe_visits_per_week', 'starbucks_visits_per_week', 'reserve_awareness_flag', 'reserve_awareness_channel', 'reserve_visited_flag', 'reserve_visit_purpose', 'reserve_service_used', 'reserve_service_reuse_flag', 'reserve_nonvisit_reason', 'reserve_ab_group', 'reserve_perception_change', 'reserve_improvement', 'reserve_recommend']

[QC] 방문 여부 분포
reserve_visited_flag
예      126
아니오     74
Name: count, dtype: int64

[QC] H4 그룹 분포
h4_group
방문 경험 있음    126
몰라서 미방문      55
알고도 미방문      19
Name: count, dtype: int64

[QC] 미방문자 수 (기대: 74)
74

[QC] 미방문 이유 분포 (원본)
reserve_nonvisit_reason
리저브 매장 존재여부 미인지(몰랐음)                   39
커피 취향/관심 낮음(단순 카페인 수혈 - 일반 매장으로 충분)    18
접근성 불편                                  9
가격 부담                                   6
매장 분위기/인테리어 불만족                         2
Name: count, dtype: int64

[QC] 미방문 이유 분포 (통합)
nonvisit_reason_grouped
인지/정보 부족    39
관심/니즈 낮음    18
접근성          9
가격           6
기타 경험요인      2
